In [1]:
import os
import sys
import torch
import cv2
import numpy as np
from tqdm import tqdm # Progress bar

# ==========================================================
# 1. PATH SETUP
# ==========================================================
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'notebooks':
    BASE_DIR = os.path.dirname(current_dir)
else:
    BASE_DIR = current_dir

# Input/Output Directories
IMAGE_DIR = os.path.join(BASE_DIR, 'src', 'data', 'imageData')
DA_OUT_DIR = os.path.join(BASE_DIR, 'src', 'data', 'da_depthmap')
MIDAS_OUT_DIR = os.path.join(BASE_DIR, 'src', 'data', 'midas_depthmap')

# Create output folders
os.makedirs(DA_OUT_DIR, exist_ok=True)
os.makedirs(MIDAS_OUT_DIR, exist_ok=True)

# Path for Depth-Anything-V2 Repo
REPO_PATH = os.path.join(BASE_DIR, 'src', 'models', 'Depth-Anything-V2')
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

from depth_anything_v2.dpt import DepthAnythingV2

# ==========================================================
# 2. INITIALIZE MODELS
# ==========================================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# --- Load Depth-Anything-V2 ---
da_config = {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]}
da_model = DepthAnythingV2(**da_config)
ckpt_path = os.path.join(REPO_PATH, 'checkpoints', 'depth_anything_v2_vitb.pth')
da_model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
da_model.to(DEVICE).eval()

# --- Load MiDaS Small ---
midas_model_type = "MiDaS_small"
midas = torch.hub.load("intel-isl/MiDaS", midas_model_type)
midas.to(DEVICE).eval()
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
midas_transform = midas_transforms.small_transform

# ==========================================================
# 3. PROCESSING LOOP
# ==========================================================
images = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Found {len(images)} images to process.")

for img_name in tqdm(images):
    # Get base ID (everything before '_image')
    # e.g., 00ABF8E5-677E..._image.jpg -> 00ABF8E5-677E...
    unique_id = img_name.split('_image')[0]
    
    # Load Image
    img_path = os.path.join(IMAGE_DIR, img_name)
    raw_image = cv2.imread(img_path)
    if raw_image is None: continue
    rgb_image = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)
    
    # --- A. Depth-Anything-V2 Inference ---
    da_depth = da_model.infer_image(raw_image, input_size=518)
    da_norm = cv2.normalize(da_depth, None, 0, 255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    da_heatmap = cv2.applyColorMap(da_norm, cv2.COLORMAP_MAGMA)
    
    # --- B. MiDaS Inference ---
    input_batch = midas_transform(rgb_image).to(DEVICE)
    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=rgb_image.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    midas_depth = prediction.cpu().numpy()
    midas_norm = cv2.normalize(midas_depth, None, 0, 255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    midas_heatmap = cv2.applyColorMap(midas_norm, cv2.COLORMAP_MAGMA)

    # ==========================================================
    # 4. SAVE WITH CUSTOM NAMING
    # ==========================================================
    # DA Filename: [ID]_dadm.png
    da_filename = f"{unique_id}_dadm.png"
    cv2.imwrite(os.path.join(DA_OUT_DIR, da_filename), da_heatmap)
    
    # MiDaS Filename: [ID]_midasdm.png
    midas_filename = f"{unique_id}_midasdm.png"
    cv2.imwrite(os.path.join(MIDAS_OUT_DIR, midas_filename), midas_heatmap)

print("\nProcessing finished! Check your da_depthmap and midas_depthmap folders.")

xFormers not available
xFormers not available


Using device: cpu


Using cache found in C:\Users\kanha/.cache\torch\hub\intel-isl_MiDaS_master
C:\Users\kanha\OneDrive\Desktop\Conda\MachLeData\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading weights:  None


Using cache found in C:\Users\kanha/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master
Using cache found in C:\Users\kanha/.cache\torch\hub\intel-isl_MiDaS_master


Found 101 images to process.


100%|████████████████████████████████████████████████████████████████| 101/101 [08:18<00:00,  4.93s/it]


Processing finished! Check your da_depthmap and midas_depthmap folders.
